# UC-STYLES-1 — Publish a Style via OGC API - Styles

**Persona:** Cartographer

**Goal:** Create a MapboxGL/GL-JS style for an existing collection, retrieve it,
update the fill colour, fetch the stylesheet body and metadata, then clean up
and open the styles browser.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `styles` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)

**References:**
- OGC API — Styles 1.0 (OGC 20-009r1)
- Routes: `/styles/catalogs/{cat}/collections/{col}/styles`

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-styles")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "land-cover")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

_style_id = None

print(f"Platform  : {BASE_URL}")
print(f"Catalog   : {CATALOG_ID}")
print(f"Collection: {COLLECTION_ID}")

## Step 1 — Create catalog and collection

Styles are scoped to a catalog/collection pair. We create a minimal demo catalog
and collection via OGC Features (409 = already exists, continue).

In [ ]:
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "Styles demo catalog"}),
)
print(f"catalog   : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({"id": COLLECTION_ID, "title": "Land cover classification"}),
)
print(f"collection: {r.status_code}", "(already exists)" if r.status_code == 409 else "")

## Step 2 — Create a style

`POST /styles/catalogs/{cat}/collections/{col}/styles` with a `StyleCreate` body.
The body carries `name`, `format`, and a `stylesheet` dict (MapboxGL JSON here).
The server returns the full `Style` record with a generated `id`.

In [ ]:
stylesheet_body = {
    "version": 8,
    "name": "Land cover — green fill",
    "sources": {
        "land-cover-src": {
            "type": "vector",
            "tiles": ["{BASE_URL}/tiles/catalogs/{catalog}/{z}/{x}/{y}.mvt"],
        }
    },
    "layers": [
        {
            "id": "land-cover-fill",
            "type": "fill",
            "source": "land-cover-src",
            "source-layer": COLLECTION_ID,
            "paint": {"fill-color": "#4caf50", "fill-opacity": 0.7},
        },
        {
            "id": "land-cover-outline",
            "type": "line",
            "source": "land-cover-src",
            "source-layer": COLLECTION_ID,
            "paint": {"line-color": "#1b5e20", "line-width": 1},
        },
    ],
}

style_payload = {
    "name": "land-cover-green",
    "title": "Land cover — green palette",
    "format": "mapbox-gl",
    "stylesheet": stylesheet_body,
    "links": [],
}

r = client.post(
    f"/styles/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/styles",
    content=json.dumps(style_payload),
)
print(f"create style: {r.status_code}")

if r.status_code == 201:
    style = r.json()
    _style_id = style["id"]
    print(f"  id      : {_style_id}")
    print(f"  name    : {style.get('name')}")
    print(f"  format  : {style.get('format')}")
    print(f"  created : {style.get('created_at')}")
else:
    print(r.text[:400])

## Step 3 — List styles for the collection

`GET /styles/catalogs/{cat}/collections/{col}/styles` returns the style list.
The new style must appear in the response.

In [ ]:
r = client.get(f"/styles/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/styles")
print(f"list styles: {r.status_code}")
if r.status_code == 200:
    styles = r.json()
    if isinstance(styles, list):
        print(f"Count: {len(styles)}")
        for s in styles:
            print(f"  id={s.get('id')}  name={s.get('name')}  format={s.get('format')}")
    else:
        print(json.dumps(styles, indent=2)[:400])
elif r.status_code == 404:
    print("SKIP: collection not found — run steps 1-2 first.")
else:
    print(r.text[:300])

## Step 4 — Update the style (change fill colour)

`PUT /styles/catalogs/{cat}/collections/{col}/styles/{style_id}` replaces the full
style record. We change the fill colour from green to earth-brown.

In [ ]:
if _style_id is None:
    print("SKIP: no style id — run step 2 first.")
else:
    updated_stylesheet = dict(stylesheet_body)
    updated_stylesheet["layers"] = [
        {
            "id": "land-cover-fill",
            "type": "fill",
            "source": "land-cover-src",
            "source-layer": COLLECTION_ID,
            "paint": {"fill-color": "#8d6e63", "fill-opacity": 0.8},
        },
        {
            "id": "land-cover-outline",
            "type": "line",
            "source": "land-cover-src",
            "source-layer": COLLECTION_ID,
            "paint": {"line-color": "#3e2723", "line-width": 1},
        },
    ]

    update_payload = {
        "name": "land-cover-brown",
        "title": "Land cover — earth palette",
        "format": "mapbox-gl",
        "stylesheet": updated_stylesheet,
        "links": [],
    }

    r = client.put(
        f"/styles/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/styles/{_style_id}",
        content=json.dumps(update_payload),
    )
    print(f"update style: {r.status_code}")
    if r.status_code == 200:
        updated = r.json()
        print(f"  name    : {updated.get('name')}")
        fill = updated.get("stylesheet", {}).get("layers", [{}])[0].get("paint", {})
        print(f"  fill-color (updated): {fill.get('fill-color')}")
    else:
        print(r.text[:300])

## Step 5 — Fetch stylesheet body and metadata

`GET /styles/catalogs/{cat}/collections/{col}/styles/{id}/stylesheet` returns the
raw MapboxGL JSON (content-negotiated). `/metadata` returns the style descriptor
with links to each available stylesheet encoding.

In [ ]:
if _style_id is None:
    print("SKIP: no style id.")
else:
    # Stylesheet body
    r = client.get(
        f"/styles/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/styles/{_style_id}/stylesheet"
    )
    print(f"stylesheet status: {r.status_code}")
    if r.status_code == 200:
        body = r.json()
        print(f"  version: {body.get('version')}")
        print(f"  name   : {body.get('name')}")
        print(f"  layers : {[l.get('id') for l in body.get('layers', [])]}")
    else:
        print(r.text[:200])

    # Metadata
    r = client.get(
        f"/styles/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/styles/{_style_id}/metadata"
    )
    print(f"\nmetadata status: {r.status_code}")
    if r.status_code == 200:
        meta = r.json()
        print(f"  id    : {meta.get('id')}")
        print(f"  name  : {meta.get('name')}")
        print(f"  links : {[l.get('rel') for l in meta.get('links', [])]}")
    else:
        print(r.text[:200])

## Teardown — Delete style, collection, catalog

In [ ]:
_TEARDOWN = os.environ.get("STYLES_TEARDOWN", "true").lower() == "true"

if _TEARDOWN:
    if _style_id:
        r = client.delete(
            f"/styles/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/styles/{_style_id}"
        )
        print(f"DELETE style     : {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
    print(f"DELETE collection: {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog   : {r.status_code}")
else:
    print("Teardown skipped (set STYLES_TEARDOWN=true to enable).")

## Result — Open the styles browser

In [ ]:
client.close()
print(f"Open the styles browser: {BASE_URL}/web/pages/styles_browser?language=en")